# Urban Growth Analysis
**NB 10** | Lagos, Nigeria 2015→2024 | Sentinel-2 | ChangeFormer

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import pygeovision as pgv
from pygeovision.models import get_model
from pygeovision.inference.tiled import TiledInference

client = pgv.PyGeoVision()
print(client)

from pygeovision.models.change_detection.changeformer import ChangeDetection


In [ ]:
BBOX        = (3.30, 6.40, 3.60, 6.65)
DATE_BEFORE = ('2015-06-01', '2015-09-30')
DATE_AFTER  = ('2024-06-01', '2024-09-30')
PROVIDERS   = ['planetary_computer']
BANDS       = ['B04','B08','B11','B12']
DATA_DIR    = pathlib.Path('./results/nb10')
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
for period, dr, outdir in [('pre', DATE_BEFORE, 'pre'), ('post', DATE_AFTER, 'post')]:
    r  = client.search(bbox=BBOX, date_range=dr, providers=PROVIDERS, cloud_cover_max=20)
    d  = r[0] if r else None
    od = DATA_DIR/outdir; od.mkdir(exist_ok=True)
    print(f"{period}: {len(r)} scenes")
    if r:
        dl = client.download([r[0]], output_dir=str(od), bands=BANDS,
            post_process=['reproject:EPSG:32632','cog'])
        globals()[f'{period}_scene'] = dl[0].path if dl and dl[0].success else None


In [ ]:
PRE_READY  = DATA_DIR / 'pre_ready.tif'
POST_READY = DATA_DIR / 'post_ready.tif'
for scene, out, label in [(pre_scene, PRE_READY,'PRE'), (post_scene, POST_READY,'POST')]:
    if scene:
        rep = client.validator.validate(str(scene), required_bands=len(BANDS))
        r   = client.prepare_for_ai(str(scene), stack_bands=BANDS, bbox=BBOX,
            normalise='scale_factor', scale_factor=10000.0,
            model_type='change_detection', output_path=str(out))
        print(f"{label}: {r['shape']}  range=[{r['array'].min():.3f},{r['array'].max():.3f}]")


In [ ]:
# NDBI on both years to quantify built-up expansion
for out_path, label in [(PRE_READY,'2015'), (POST_READY,'2024')]:
    if out_path.exists():
        ndbi_path = DATA_DIR/f'ndbi_{label}.tif'
        client.indices.ndbi(str(out_path), swir1_band=3, nir_band=2, output_path=str(ndbi_path))
        import rasterio
        with rasterio.open(str(ndbi_path)) as src: a = src.read(1)
        built = (a > 0.05).mean() * 100
        print(f"Built-up {label}: {built:.1f}% of study area (NDBI > 0.05)")


In [ ]:
CHANGE_MAP = DATA_DIR / 'urban_growth.tif'
if PRE_READY.exists() and POST_READY.exists():
    cd = ChangeDetection(model_variant='changeformer', in_channels=len(BANDS))
    cd.build()
    res = cd.detect(str(PRE_READY), str(POST_READY), output_path=str(CHANGE_MAP))
    print(f"Urban expansion: {res['change_pct']:.2f}%")


In [ ]:
SIEVED  = DATA_DIR / 'sieved.tif'
VECTOR  = DATA_DIR / 'output.geojson'
COG_OUT = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=20, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR),
        target_class=1, min_area_m2=500.0, simplify_tolerance=2.0)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Polygons detected : {len(gj['features'])}")
    for cls, info in sorted(stats.items()):
        print(f"  Class {cls}: {info['pixels']:>8,} px | {info['area_ha']:>8.1f} ha | {info['pct']:>6.1f}%")
    print(f"COG saved: {COG_OUT}")


**NB10 — Urban Growth** complete.
- Study area: Lagos Nigeria
- Sensor: Sentinel-2 2015→2024
- Model: ChangeFormer
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG